# RAG Pipeline — Evaluation & Failure Analysis (Real Run Results)

Corpus: 10 heist/action-thriller movie Wikipedia pages. Pipeline: `WikipediaLoader`
→ `RecursiveCharacterTextSplitter` (baseline chunk_size=500/overlap=50) → `Chroma`
+ `text-embedding-3-small` → explicit `PromptTemplate` → top-3 retriever →
`ChatOpenAI(gpt-4o-mini)`. These results are from an actual executed run.



## 1. Ground-truth mapping

| # | Question | Correct source passage |
|---|----------|--------------------------|
| 1 | Who leads the crew in Ocean's Eleven and what is the target of the heist? | *Ocean's Eleven* — plot summary paragraph (Danny Ocean & Rusty Ryan plan to steal $160M from Terry Benedict) |
| 2 | What is the central conflict between the two main characters in Heat? | *Heat (1995 film)* — "Plot" section describing the cop (Hanna)/thief (McCauley) cat-and-mouse dynamic |
| 3 | How does the bank robbery scheme work in Inside Man? | *Inside Man* — "Plot" section describing Dalton Russell's hostage/vault scheme |
| 4 | What role does music/soundtrack play in Baby Driver's car chase sequences? | *Baby Driver* — "Production"/"Style" section on the music-synced editing and stunt choreography |
| 5 | What is the Professor's plan in Money Heist and what does RTVE's coverage/reception say about the show? | *Money Heist* — "Plot" section (mint takeover) **and** a separate "Reception"/broadcast section mentioning RTVE |



## 2. Retrieval scoring table (from actual run)

| # | Question | Top-3 contains correct info? (Y/N) | Rank of first relevant chunk |
|---|----------|:---:|:---:|
| 1 | Ocean's Eleven crew/target | **Y** | 2 |
| 2 | Heat's central conflict | **N** | — |
| 3 | Inside Man robbery scheme | **N** | — |
| 4 | Baby Driver soundtrack role | **N** | — |
| 5 | Money Heist Professor's plan / RTVE reception | **Partial** — plan half Y (rank 3), reception half N | 3 (plan only) |

**Retrieval Success Rate = 1 / 5 = 0.20** (counting Q5 as a failure since the
question was only half-answered; 1.5/5 = 0.30 if partial credit is allowed).



## 3. What actually happened, question by question

- **Q1 (success):** Chunk 2 of *Ocean's Eleven* directly stated "Danny Ocean
  (Clooney) and Rusty Ryan (Pitt), who plan a heist of $160 million from
  casino owner Terry Benedict" — an almost verbatim match to the question, so
  both similarity search and the LLM performed as intended.
- **Q2 (failure):** All 3 retrieved chunks for *Heat* were generic
  (production history, box office, and the opening synopsis line), never
  reaching the actual antagonist-dynamic description in the "Plot" section.
  The LLM correctly declined to answer rather than hallucinate.
- **Q3 (failure):** Two of the three chunks were **identical duplicates** of
  the *Inside Man* opening synopsis; the third came from the wrong movie
  (*Den of Thieves*) because "bank robbery scheme" is generic enough to be
  semantically close to multiple heist-movie chunks in the shared vector
  store. The real robbery-mechanics text (twist ending, decoy vault) was
  never retrieved.
- **Q4 (failure, in both configs):** All three chunks across both the
  baseline (500/50) and alt config (300/100) were near-duplicates of the
  intro paragraph. Re-chunking with a smaller size did **not** fix this —
  the "Style"/music-synced-editing content it needed lives further down the
  page than any of the top-3 nearest-neighbor chunks in embedding space.
- **Q5 (partial):** The Professor's-plan half was answered correctly (mint
  takeover, 8 people, €984M, 11 days) — that content sat in chunk 3.
  The RTVE/reception half was never retrieved at all; no chunk mentioning
  broadcast reception made the top-3.



## 4. Failure analysis (3 documented cases)

**Failure 1 — Retrieval returns near-duplicate chunks instead of diverse
coverage (Q3, Q5).**
Category: *irrelevant/redundant chunk ranked highly.*
For both *Inside Man* and *Money Heist*, chunk 1 and chunk 2 in the top-3
were textually identical (same opening paragraph, retrieved twice — likely
because the splitter produced overlapping duplicate-looking chunks near a
paragraph boundary, and both scored nearly the same cosine similarity to the
query). This wastes 1 of 3 available retrieval slots on redundant text
instead of surfacing a second, complementary passage. **Fix:** deduplicate
retrieved chunks by content hash before building the context, or use
maximal-marginal-relevance (MMR) search (`search_type="mmr"`) instead of
plain top-k similarity, which explicitly penalizes near-duplicate results.

**Failure 2 — Vector store mixes movies, causing cross-document leakage
(Q3).**
Category: *ambiguous/cross-document retrieval confusion.*
A generic phrase like "bank robbery scheme" is semantically close to chunks
from *multiple* heist films in the same collection. The retriever pulled a
chunk from *Den of Thieves* when the question was specifically about
*Inside Man*. This is the multi-movie-corpus version of the "ambiguous movie
name" failure mode: even with an unambiguous, correctly-loaded document set,
a shared vector store across similar-genre documents lets off-topic but
semantically similar content crowd out the right document's chunk.
**Fix:** add a metadata filter (retrieve only within the named movie's
`source`) when the question names a specific film, or retrieve more
candidates (k=5–8) and re-rank by keyword overlap with the named entity
before truncating to the top 3.

**Failure 3 — Chunking granularity doesn't help when the relevant fact is
simply low-ranked in embedding space (Q4).**
Category: *relevant chunk ranked too low / correct document not retrieved.*
Re-running Q4 with a smaller chunk size (300/100 vs. 500/50) did not change
the outcome at all — both configs retrieved the same generic intro content.
This shows chunk size tuning only helps when the problem is *boundary
splitting* (a fact cut in half); it does nothing when the real issue is that
the query's embedding is simply closer to the introduction than to the
specific section containing the answer. **Fix:** this needs a retrieval
strategy change, not a chunking change — e.g., query rewriting/expansion
("Baby Driver stunt choreography music editing style" instead of the raw
question), hybrid keyword+vector search, or increasing k so the correct
chunk has more chances to appear even if it isn't in the very top 3.



## 5. Chunking comparison (Q1 and Q4 re-run at chunk_size=300, overlap=100)

| Question | Baseline (500/50) top chunk | Alt (300/100) top chunk | Changed? | Answer changed? |
|---|---|---|---|---|
| Q1 (Ocean's Eleven) | Intro paragraph | Same intro paragraph | No | No — correct both times |
| Q4 (Baby Driver) | Intro + reception paragraphs (duplicated) | Same intro paragraph, still duplicated | No | No — still "not in context" both times |

**Takeaway:** smaller chunks / more overlap chunking, alone, fixed nothing here.
The bottleneck for the failing questions is *retrieval ranking*, not chunk
boundaries — reinforcing failure case 3 above. This is a useful, honest
finding for the "compare configs" part of the assignment: not every retrieval
problem is a chunking problem.
